Downloading Libraries and Imports

In [1]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade "pillow<11.0.0" torch torchvision transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv


!pip uninstall -y torchaudio -q


!pip install -q flash-linear-attention

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


W0921 20:03:46.714000 4041 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0921 20:03:46.759000 4041 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Setting up environment

In [2]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "Ivo",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        # /content survives "Restart session", so an old clone can linger -- pull so
        # the notebook always runs the latest code (restart the session afterwards if
        # eval.utilities was already imported in this kernel).
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_1,
   preprocess_function,
   get_patch_reordering_indices,
   reorder_chessboard_image,
   apply_patch_permutation,
   finetune_and_push_chessboard_model,
   find_resumable_checkpoint,
   Qwen35OnTheFlyCollator,
)

from training.learned_reordering import train_with_learned_reordering

print("All custom modules and eval utilities imported successfully!")

Cloning repository from https://github.com/Aivon99/BigDataAndTextMiningProject.git...
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
All custom modules and eval utilities imported successfully!


Dowloading dataset for task 1 from HuggingFace repo

In [3]:
# Authenticate with Hugging Face:
# - Colab: reads a token from a Colab secret named HF_TOKEN (key icon, left sidebar)
# - Local: reads HF_TOKEN from a repo-root .env file, or the environment
# - Falls back to an interactive login prompt if neither is found
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment — falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name, num_proc=16)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Logged in to Hugging Face Hub using HF_TOKEN.


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

metadata.jsonl:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

...

Setting num_proc from 16 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


Generating validation split:   0%|          | 0/400 [00:00<?, ? examples/s]

Setting num_proc from 16 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': 'drw7g', 'task': 'task1', 'fen': 'r3k2r/pppq2pp/2nb1p2/1B1Q3P/8/1P2P2P/PBPP1P2/R3K2R w KQkq - 1 13', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of a

## Vanilla Model

Loading model

In [4]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

Loading model Qwen/Qwen3.5-0.8B...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

...

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model and Processor loaded correctly!


Test with baseline on a single sample

In [5]:
# Grab the first test sample
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

Sample ID: sample_000000
FEN: 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to extract the exact board state from the provided chessboard image.
Input:
- Board Image: The visual representation of the chessboard.
Output Format:
Return only the valid FEN string representing the position of all pieces on the board.

Real FEN (Ground Truth): 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33

Generating zero-shot prediction...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Predicted FEN (Zero-Shot): e2r1qpppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppp


Testing vanilla model on the whole dataset using the function *evaluate_chessboard_model_task_1*

In [6]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Vanilla Qwen2.5-VL (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head())

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)

Evaluating Vanilla Qwen2.5-VL (Zero-Shot): 100%|██████████| 400/400 [29:43<00:00,  4.46s/it]


Evaluation completed for Vanilla Qwen2.5-VL (Zero-Shot)! Results saved to task1_vanilla_qwen2.5-vl_(zero-shot)_results.csv.

Example of results obtained form the evaluation:


,sample_id,ground_truth,predicted,raw_output,fen_exact_match,levenshtein_distance,character_error_rate,square_by_square_accuracy
0,sample_000000,6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - -...,e2r1qppppppppppppppppppppppppppppppppppppppppp...,e2r1qppppppppppppppppppppppppppppppppppppppppp...,0.0,190,3.725490,0.0
1,sample_000001,r1r5/p6n/4pq2/3P1p1k/7P/5PQ1/P5P1/R3R1K1 w - -...,b1r1p1pppppppppppppppppppppppppppppppppppppppp...,b1r1p1pppppppppppppppppppppppppppppppppppppppp...,0.0,189,3.705882,0.0
2,sample_000002,8/6pk/5np1/1p5p/p1pP3q/P1P1Q3/1P3P2/5RK1 w - -...,b1rnb1q1ppp1pppppppppppppppppppppppppppppppppp...,b1rnb1q1ppp1pppppppppppppppppppppppppppppppppp...,0.0,186,3.647059,0.0
3,sample_000003,r4r1k/6p1/p5P1/1p2ppNP/1q2b2Q/4n3/PPPRB3/1K4R1...,a1b1c1d1e1f1g1h1i1j1k1l1m1n1o1p1q1r1s1t1u1v1w1...,a1b1c1d1e1f1g1h1i1j1k1l1m1n1o1p1q1r1s1t1u1v1w1...,0.0,49,0.859649,0.0
4,sample_000004,4rrk1/6pp/p1p1q3/3p2Q1/2PP2b1/1PR2N2/P5PP/5RK1...,e2r1q1ppp1pppppppppppppppppppppppppppppppppppp...,e2r1q1ppp1pppppppppppppppppppppppppppppppppppp...,0.0,185,3.245614,0.0



Comparative Summary DataFrame (all_models_results):


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.0,169.225


## Vanilla + LoRA

Preprocessing dataset and finetuning

In [7]:
# 1. No dataset.map preprocessing: the collator below tokenizes and runs the
# image processor per batch (in DataLoader workers), so nothing big is cached.

# 2. Configure PEFT and LoRA parameters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to model vanilla and saving the new one into lora_model
lora_model = get_peft_model(model_vanilla, peft_config)
lora_model.print_trainable_parameters()

# 3. Resolve the target Hub repo now (needed before training starts) and
# check whether an earlier, interrupted run already left a resumable
# checkpoint there (e.g. after a Colab disconnect wiped the local runtime).
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task1-standard-lora"
resume_checkpoint = find_resumable_checkpoint(repo_id_standard)

# 4. Define Training Arguments. Trains for up to 10 epochs, but relies on
# the validation set (via EarlyStoppingCallback below) to stop once
# eval_loss stops improving, and keeps the best-performing checkpoint
# (by validation loss) rather than just whichever epoch finishes last.
# push_to_hub + hub_strategy="checkpoint" uploads a fully resumable
# checkpoint (optimizer/scheduler/RNG state included) after every epoch,
# so a Colab disconnect at epoch i loses at most that epoch's progress.
training_args = TrainingArguments(
    output_dir="./qwen_task1_lora_output",
    per_device_train_batch_size=8,  # same effective batch (8) as before, but one GPU pass instead of 8
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=repo_id_standard,
    hub_strategy="checkpoint",
    bf16=True,  # model is loaded in bfloat16; fp16 autocast on top of it was mismatched
    dataloader_num_workers=4,
    remove_unused_columns=False,
    report_to="none",
)

# 5. Initialize the Trainer using 'lora_model', with early stopping driven
# by the validation set (stops if eval_loss doesn't improve for 2 epochs).
# data_collator preprocesses each batch on the fly (image processor + tokenizer)
# rather than precomputing pixel_values with dataset.map -- that cache would be
# ~20 GB of float tensors and slow to reload -- and it batches Qwen's per-sample
# image_grid_thw/pixel_values correctly (torch.cat along dim 0), which the
# default collator can't (it fails later inside the vision tower as IndexError).
data_collator = Qwen35OnTheFlyCollator(processor=processor)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=dataset_task1["train"],
    eval_dataset=dataset_task1["validation"],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# 6. Start (or resume) Fine-Tuning
if resume_checkpoint:
    print(f"Resuming LoRA Supervised Fine-Tuning from {resume_checkpoint}...")
else:
    print("Starting LoRA Supervised Fine-Tuning...")
trainer.train(resume_from_checkpoint=resume_checkpoint)

trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 0 files: 0it [00:00, ?it/s]

Starting LoRA Supervised Fine-Tuning...


Epoch,Training Loss,Validation Loss
1,2.382977,2.382412
2,2.367021,2.368236
3,2.357597,2.358504
4,2.347972,2.364700
5,2.352039,2.355937
6,2.331720,2.356646
7,2.315592,2.341038
8,2.304512,2.354887
9,2.286807,2.357151


TrainOutput(global_step=3600, training_loss=2.3611699353324043, metrics={'train_runtime': 1446.0184, 'train_samples_per_second': 22.13, 'train_steps_per_second': 2.766, 'total_flos': 4.065660124633498e+16, 'train_loss': 2.3611699353324043, 'epoch': 9.0})

Saving model on hugging face

In [8]:
# 6. Save weights to Hugging Face folder
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task1-standard-lora"

print(f"Pushing standard LoRA model and processor to Hugging Face Hub: {repo_id_standard}...")

trainer.model.push_to_hub(
    repo_id_standard,
    commit_message="Training complete for standard LoRA baseline (raster-scan)"
)
processor.push_to_hub(
    repo_id_standard
)

print("Fine-tuning completed and weights successfully uploaded to Hugging Face Hub!")

Pushing standard LoRA model and processor to Hugging Face Hub: bdatm-project/qwen-task1-standard-lora...


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  62%|######2   | 15.9MB / 25.6MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpqkd5kx_6/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Fine-tuning completed and weights successfully uploaded to Hugging Face Hub!


Loading model and evaluation

In [9]:
# 7. Loading Model from hugging face and evaluate model using predefined functions
print("\nLoading standard LoRA model from Hugging Face for evaluation...")

# Load fresh base model instance
base_eval_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Load the LoRA weights directly from the Hub repository
standard_lora_eval_model = PeftModel.from_pretrained(base_eval_model, repo_id_standard)

print("Evaluating the Hugging Face LoRA model on the test set...")
lora_results_df, lora_summary_df = evaluate_chessboard_model_task_1(
    model=standard_lora_eval_model,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Qwen + LoRA Fine-Tuning (from HF)"
)

# Append the new metrics to the global comparison DataFrame
all_models_results = pd.concat([all_models_results, lora_summary_df], ignore_index=True)

print("\nUpdated Comparative Summary Table (all_models_results):")
display(all_models_results)


Loading standard LoRA model from Hugging Face for evaluation...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Evaluating the Hugging Face LoRA model on the test set...


Evaluating Qwen + LoRA Fine-Tuning (from HF): 100%|██████████| 400/400 [23:23<00:00,  3.51s/it]


Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task1_qwen_+_lora_fine-tuning_(from_hf)_results.csv.

Updated Comparative Summary Table (all_models_results):


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.000000,169.2250
1,Qwen + LoRA Fine-Tuning (from HF),0.0,0.474183,0.691133,25.1425


## Reordering patches - Advanced models

Checking function *get_patch_reordering_indices()*

In [10]:
for strat in ["raster", "zigzag", "spiral", "file_wise"]:
    order_map = get_patch_reordering_indices(strategy=strat)
    print(f"Strategy '{strat}' first 10 patch indices: {order_map[:10]}")

Strategy 'raster' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Strategy 'zigzag' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 15, 14]
Strategy 'spiral' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 15, 23]
Strategy 'file_wise' first 10 patch indices: [0, 8, 16, 24, 32, 40, 48, 56, 1, 9]


Let's evalaute the model we created before using three different patch ordering: zigzag, spiral and file-wise. We'll use the function ***reorder_chessboard_image*** defined in *src/eval/utilities.py*

In [11]:
# ==========================================
# Training-Free Benchmark Loop for REOrder
# ==========================================

# Define the strategies you want to benchmark (as outlined in the project specs)
strategies_to_test = ["zigzag", "spiral", "file_wise"]

# Choose the model to test (we use the fine-tuned LoRA model)
model_to_evaluate = lora_model  # You can switch to 'model' if you want to test the vanilla version

for strat in strategies_to_test:
    print(f"\nEvaluating strategy (Training-Free): {strat.upper()}...")

    # 1. Apply the reordering function to the images in the test set
    reordered_test_split = dataset_task1["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 2. Run the evaluation function on the reordered test split
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_1(
        model=model_to_evaluate,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - TF)"
    )

    # 3. Append the results to your global comparison table
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table (Including Training-Free Strategies) ---")
display(all_models_results)


Evaluating strategy (Training-Free): ZIGZAG...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (Zigzag - TF): 100%|██████████| 400/400 [23:06<00:00,  3.47s/it]


Evaluation completed for Qwen + LoRA (Zigzag - TF)! Results saved to task1_qwen_+_lora_(zigzag_-_tf)_results.csv.

Evaluating strategy (Training-Free): SPIRAL...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

...


Evaluation completed for Qwen + LoRA (Spiral - TF)! Results saved to task1_qwen_+_lora_(spiral_-_tf)_results.csv.

Evaluating strategy (Training-Free): FILE_WISE...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (File_wise - TF): 100%|██████████| 400/400 [24:05<00:00,  3.61s/it]


Evaluation completed for Qwen + LoRA (File_wise - TF)! Results saved to task1_qwen_+_lora_(file_wise_-_tf)_results.csv.

--- Final Comparative Summary Table (Including Training-Free Strategies) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.000000,169.2250
1,Qwen + LoRA Fine-Tuning (from HF),0.0,0.474183,0.691133,25.1425
2,Qwen + LoRA (Zigzag - TF),0.0,0.541779,0.633008,28.7325
3,Qwen + LoRA (Spiral - TF),0.0,0.568413,0.619219,30.1100
4,Qwen + LoRA (File_wise - TF),0.0,0.594664,0.597305,31.6225


As highlighted in the summary table, applying unconventional patch reordering strategies (such as Zigzag, Spiral, or File-wise) in a "Training-Free" (TF) manner leads to a performance drop, resulting in an increased Character Error Rate (CER) and Levenshtein distance compared to the standard raster-scan baseline.

To truly reap the benefits of the REOrder methodology, we must proceed with Supervised Fine-Tuning (SFT) directly on the pre-reordered dataset. This will allow the model to adapt its weights and attention layers to the new spatial serialization strategy.

Finetuning the new models on the dataset using function **finetune_and_push_chessboard_model()** defined in *src/eval/utilities.py*. This function directly upload the models on hugging face

In [12]:
strategies_to_train = ["zigzag", "spiral", "file_wise"]

# Dictionary to store the trained models in memory
trained_reordered_models = {}

print("=== Starting Fine-Tuning Pipeline for Task 1 (Reordering Strategies) ===")

for strat in strategies_to_train:
    print(f"\nLoading fresh base model for Task 1 | Strategy: {strat}...")

    # Load a clean instance of the base model for each strategy
    base_model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    trained_reordered_models[strat] = finetune_and_push_chessboard_model(
        strategy_name=strat,
        dataset=dataset_task1,
        processor=processor,
        model=base_model,
        peft_config=peft_config,
        task="task1",
        hf_org_prefix=hf_org_prefix,
    )

print("\nAll reordered models have been successfully trained and pushed to Hugging Face!")

=== Starting Fine-Tuning Pipeline for Task 1 (Reordering Strategies) ===

Loading fresh base model for Task 1 | Strategy: zigzag...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


Starting pipeline for TASK: TASK1 | STRATEGY: ZIGZAG
Applying LoRA to the provided model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 0 files: 0it [00:00, ?it/s]

...

  ...adapter_model.safetensors:  62%|######2   | 15.9MB / 25.6MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp2g534osr/tokenizer.json:  80%|#######9  | 16.0MB / 20.0MB            

Finished! Successfully uploaded to Hub: bdatm-project/qwen-task1-file_wise-lora

All reordered models have been successfully trained and pushed to Hugging Face!


Evaluating models using function **evaluate_chessboard_model_task_1()** defined in *src/eval/utilities.py*.

Models are downloaded from the hugging face repo.

In [13]:
strategies_to_evaluate = ["zigzag", "spiral", "file_wise"]
hf_org_prefix = "bdatm-project"  # Assicurati che corrisponda al prefisso usato per il push

for strat in strategies_to_evaluate:
    print(f"\nLoading and evaluating SFT model for strategy: {strat.upper()} from Hugging Face...")

    # 1. Load fresh base model instance
    base_eval_model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3.5-0.8B",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # 2. Load the specific LoRA weights from Hugging Face Hub
    repo_id_source = f"{hf_org_prefix}/qwen-task1-{strat}-lora"
    model_to_eval = PeftModel.from_pretrained(base_eval_model, repo_id_source)

    # 3. Apply the specific patch reordering to the test split images
    reordered_test_split = dataset_task1["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 4. Run the evaluation utility function
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_1(
        model=model_to_eval,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - SFT)"
    )

    # 5. Append results to the global comparison DataFrame
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table (Loaded from Hub & Evaluated) ---")
display(all_models_results)


Loading and evaluating SFT model for strategy: ZIGZAG from Hugging Face...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Evaluating Qwen + LoRA (File_wise - SFT): 100%|██████████| 400/400 [25:25<00:00,  3.81s/it]


Evaluation completed for Qwen + LoRA (File_wise - SFT)! Results saved to task1_qwen_+_lora_(file_wise_-_sft)_results.csv.

--- Final Comparative Summary Table (Loaded from Hub & Evaluated) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.000000,169.2250
1,Qwen + LoRA Fine-Tuning (from HF),0.0,0.474183,0.691133,25.1425
2,Qwen + LoRA (Zigzag - TF),0.0,0.541779,0.633008,28.7325
3,Qwen + LoRA (Spiral - TF),0.0,0.568413,0.619219,30.1100
4,Qwen + LoRA (File_wise - TF),0.0,0.594664,0.597305,31.6225
5,Qwen + LoRA (Zigzag - SFT),0.0,0.582808,0.622969,30.9100
6,Qwen + LoRA (Spiral - SFT),0.0,0.515312,0.654766,27.3200
7,Qwen + LoRA (File_wise - SFT),0.0,0.585370,0.619492,30.9950


## Learned Reordering (Project Work)

Instead of a fixed strategy, trains a lightweight `PlackettLucePatchPolicy` (`src/training/learned_reordering.py`) jointly with LoRA to learn which 8x8 patch order helps the model most, via REINFORCE using the per-sample LM loss as the reward — see the module's docstring for how this differs from REOrder's own from-scratch-model approach.

**Known limitations of this first version, worth knowing before running:** unlike the `Trainer`-based cells above, this custom training loop has **no checkpointing or resume support** — a Colab disconnect mid-run loses all progress, not just the current epoch. It's also unbatched (one sample at a time, no gradient accumulation) and does two backward passes per sample (model + policy), so it's slower per-epoch than the standard fine-tuning above. `num_train_epochs` is kept low below (3, not the default 10) to keep a first run's wall-clock time and disconnect risk bounded — raise it once you've confirmed this runs end-to-end.

In [14]:
print("=== Starting Learned Reordering Training Pipeline for Task 1 ===")

# Fresh base model instance, same pattern as the training-free strategies above
base_model_learned = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

learned_lora_model, learned_policy, repo_id_learned = train_with_learned_reordering(
    dataset=dataset_task1,
    processor=processor,
    model=base_model_learned,
    peft_config=peft_config,
    task="task1",
    hf_org_prefix=hf_org_prefix,
    num_train_epochs=3,  # kept low for a first run -- see the note above
)

print(f"\nLearned Reordering training complete. Model pushed to: {repo_id_learned}")

=== Starting Learned Reordering Training Pipeline for Task 1 ===


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] epoch 0 step 10: lm_loss=6.2071 policy_loss=-87.3584 baseline=-5.7808
[task1] epoch 0 step 20: lm_loss=3.4059 policy_loss=261.8914 baseline=-4.7160
[task1] epoch 0 step 30: lm_loss=2.7420 policy_loss=165.4492 baseline=-3.5493
[task1] epoch 0 step 40: lm_loss=2.5647 policy_loss=77.4861 baseline=-2.9444
[task1] epoch 0 step 50: lm_loss=2.5213 policy_loss=30.7993 baseline=-2.6733
[task1] epoch 0 step 60: lm_loss=2.5180 policy_loss=10.8298 baseline=-2.5711
[task1] epoch 0 step 70: lm_loss=2.4610 policy_loss=10.2045 baseline=-2.5109
[task1] epoch 0 step 80: lm_loss=2.5034 policy_loss=-2.7113 baseline=-2.4901
[task1] epoch 0 step 90: lm_loss=2.4842 policy_loss=-1.2460 baseline=-2.4781
[task1] epoch 0 step 100: lm_loss=2.4626 policy_loss=-0.2665 baseline=-2.4613
[task1] epoch 0 step 110: lm_loss=2.4046 policy_loss=10.5776 baseline=-2.4566
[task1] epoch 0 step 120: lm_loss=2.4448 policy_loss=0.1946 baseline=-2.4458
[task1] epoch 0 step 130: lm_loss=2.4089 policy_loss=5.6893 baseline=-2

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp3w8beglc/tokenizer.json:  80%|#######9  | 16.0MB / 20.0MB            

Saved learned reordering policy weights to learned_reordering_output/policy.pt

Learned Reordering training complete. Model pushed to: bdatm-project/qwen-task1-learned-reordering-lora


Evaluating the learned-reordering model: training samples permutations stochastically (that's how REINFORCE explores), but evaluation needs one fixed ordering — `policy.greedy_permutation()` gives the policy's single best guess (argsort of its learned logits, no sampling noise), which is what actually gets applied to the test set here.

In [15]:
print("=== Evaluating the Learned Reordering model on Task 1 ===")

learned_permutation = learned_policy.greedy_permutation()
print(f"Learned (greedy) patch order: {learned_permutation}")

reordered_test_split = dataset_task1["test"].map(
    lambda sample: {
        "image": apply_patch_permutation(sample["image"], learned_permutation, grid_size=8)
    }
)

learned_results_df, learned_summary_df = evaluate_chessboard_model_task_1(
    model=learned_lora_model,
    processor=processor,
    dataset_split=reordered_test_split,
    model_name="Qwen + LoRA (Learned Reordering)"
)

all_models_results = pd.concat([all_models_results, learned_summary_df], ignore_index=True)

print("\n--- Comparative Summary Table (Including Learned Reordering) ---")
display(all_models_results)

=== Evaluating the Learned Reordering model on Task 1 ===
Learned (greedy) patch order: [14, 26, 3, 32, 1, 16, 19, 18, 45, 41, 15, 25, 9, 35, 5, 4, 23, 40, 11, 29, 22, 13, 8, 49, 44, 50, 53, 58, 24, 7, 2, 12, 39, 37, 62, 0, 51, 17, 34, 47, 38, 33, 28, 31, 10, 57, 30, 43, 20, 21, 63, 59, 48, 36, 46, 6, 56, 27, 42, 60, 54, 52, 55, 61]


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (Learned Reordering): 100%|██████████| 400/400 [28:24<00:00,  4.26s/it]


Evaluation completed for Qwen + LoRA (Learned Reordering)! Results saved to task1_qwen_+_lora_(learned_reordering)_results.csv.

--- Comparative Summary Table (Including Learned Reordering) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.000000,169.2250
1,Qwen + LoRA Fine-Tuning (from HF),0.0,0.474183,0.691133,25.1425
2,Qwen + LoRA (Zigzag - TF),0.0,0.541779,0.633008,28.7325
3,Qwen + LoRA (Spiral - TF),0.0,0.568413,0.619219,30.1100
4,Qwen + LoRA (File_wise - TF),0.0,0.594664,0.597305,31.6225
5,Qwen + LoRA (Zigzag - SFT),0.0,0.582808,0.622969,30.9100
6,Qwen + LoRA (Spiral - SFT),0.0,0.515312,0.654766,27.3200
7,Qwen + LoRA (File_wise - SFT),0.0,0.585370,0.619492,30.9950
8,Qwen + LoRA (Learned Reordering),0.0,0.564163,0.634336,30.0175


Pushing Task 1 evaluation results to Hugging Face, so they can be reloaded later without re-running any inference (same approach used in *main.ipynb*)

In [16]:
print("=== Pushing Task 1 Evaluation Results to Hugging Face ===")

hf_dataset_results_task1 = Dataset.from_pandas(all_models_results)
repo_id_eval_task1 = f"{hf_org_prefix}/evaluation-results-task1"

print(f"Pushing Task 1 evaluation results to Hugging Face Hub: {repo_id_eval_task1}...")
hf_dataset_results_task1.push_to_hub(
    repo_id_eval_task1,
    private=False
)
print("Task 1 evaluation results successfully pushed to Hugging Face Hub!")

=== Pushing Task 1 Evaluation Results to Hugging Face ===
Pushing Task 1 evaluation results to Hugging Face Hub: bdatm-project/evaluation-results-task1...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpk8amympy.parquet    : 100%|##########| 3.33kB / 3.33kB            

Task 1 evaluation results successfully pushed to Hugging Face Hub!


Displaying oredered results

In [17]:
print("=== Downloading Task 1 Evaluation Results from Hugging Face ===")
downloaded_eval_results_task1 = load_dataset(repo_id_eval_task1, split="train").to_pandas()

print("\n--- TASK 1 Comparative Summary Table (Ordered from Best to Worst by CER, loaded from Hugging Face) ---")
sorted_results_task1_from_hub = downloaded_eval_results_task1.sort_values(
    by="character_error_rate", ascending=True
).reset_index(drop=True)
display(sorted_results_task1_from_hub)

=== Downloading Task 1 Evaluation Results from Hugging Face ===


README.md:   0%|          | 0.00/466 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.33kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9 [00:00<?, ? examples/s]


--- TASK 1 Comparative Summary Table (Ordered from Best to Worst by CER, loaded from Hugging Face) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Qwen + LoRA Fine-Tuning (from HF),0.0,0.474183,0.691133,25.1425
1,Qwen + LoRA (Spiral - SFT),0.0,0.515312,0.654766,27.3200
2,Qwen + LoRA (Zigzag - TF),0.0,0.541779,0.633008,28.7325
3,Qwen + LoRA (Learned Reordering),0.0,0.564163,0.634336,30.0175
4,Qwen + LoRA (Spiral - TF),0.0,0.568413,0.619219,30.1100
5,Qwen + LoRA (Zigzag - SFT),0.0,0.582808,0.622969,30.9100
6,Qwen + LoRA (File_wise - SFT),0.0,0.585370,0.619492,30.9950
7,Qwen + LoRA (File_wise - TF),0.0,0.594664,0.597305,31.6225
8,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.000000,169.2250


## Results and Final Considerations

Reloading the Task 1 evaluation results directly from Hugging Face (instead of from the in-memory `all_models_results`), to confirm they persist independently of this notebook session — same idea as the "Results and Final Considerations" section in *main.ipynb*.

In [18]:
sorted_results_df = all_models_results.sort_values(by="character_error_rate", ascending=True).reset_index(drop=True)

print("\n--- Comparative Summary Table (Ordered from Best to Worst by CER) ---")
display(sorted_results_df)


--- Comparative Summary Table (Ordered from Best to Worst by CER) ---


,model_name,fen_exact_match,character_error_rate,square_by_square_accuracy,levenshtein_distance
0,Qwen + LoRA Fine-Tuning (from HF),0.0,0.474183,0.691133,25.1425
1,Qwen + LoRA (Spiral - SFT),0.0,0.515312,0.654766,27.3200
2,Qwen + LoRA (Zigzag - TF),0.0,0.541779,0.633008,28.7325
3,Qwen + LoRA (Learned Reordering),0.0,0.564163,0.634336,30.0175
4,Qwen + LoRA (Spiral - TF),0.0,0.568413,0.619219,30.1100
5,Qwen + LoRA (Zigzag - SFT),0.0,0.582808,0.622969,30.9100
6,Qwen + LoRA (File_wise - SFT),0.0,0.585370,0.619492,30.9950
7,Qwen + LoRA (File_wise - TF),0.0,0.594664,0.597305,31.6225
8,Vanilla Qwen2.5-VL (Zero-Shot),0.0,3.259093,0.000000,169.2250
